In [35]:
import requests
from bs4 import BeautifulSoup
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F


In [36]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
model_name = "roberta-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Device set to use cpu
Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [70]:
def fetch_text_from_url(url):
    """Fetches and extracts main text from a webpage."""
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        res = requests.get(url, timeout=10, headers=headers)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")

        for element in soup(["script", "style", "nav", "footer", "header", "aside", "form"]):
            element.decompose()

        text_parts = []

        content_selectors = [
            'main', 'article', '[role="main"]',
            '.content', '.main-content', '.post-content',
            '.entry-content', '.article-content', '.story-body'
        ]

        main_content = None
        for selector in content_selectors:
            main_content = soup.select_one(selector)
            if main_content:
                break

        if main_content:
            paragraphs = main_content.find_all(["p", "div", "span", "h1", "h2", "h3", "h4", "h5", "h6"])
            for p in paragraphs:
                text = p.get_text().strip()
                if text and len(text) > 20:  # Only include substantial text
                    text_parts.append(text)
        else:
            elements = soup.find_all(["p", "h1", "h2", "h3", "h4", "h5", "h6"])
            for element in elements:
                text = element.get_text().strip()
                if text and len(text) > 20:  # Only include substantial text
                    text_parts.append(text)

        if not text_parts:
            divs = soup.find_all("div")
            for div in divs:
                text = div.get_text().strip()
                if text and len(text.split()) > 10:  # At least 10 words
                    text_parts.append(text)

        full_text = " ".join(text_parts)

        import re
        full_text = re.sub(r'\s+', ' ', full_text)
        sentences = full_text.split('.')
        meaningful_sentences = [s.strip() for s in sentences if len(s.strip().split()) > 3]

        return '. '.join(meaningful_sentences).strip()

    except Exception as e:
        print(f"Failed to fetch from {url}: {e}")
        return ""


In [71]:
def simple_text_truncation(text, max_words=500):
    """Simple text truncation that preserves sentence boundaries."""
    words = text.split()
    if len(words) <= max_words:
        return text


    truncated = " ".join(words[:max_words])

    last_period = truncated.rfind('.')
    last_exclamation = truncated.rfind('!')
    last_question = truncated.rfind('?')

    last_sentence_end = max(last_period, last_exclamation, last_question)

    if last_sentence_end > len(truncated) * 0.7:
        return truncated[:last_sentence_end + 1]
    else:
        return truncated + "..."

In [72]:
def summarize_text(text, max_len=1000):
    if not text or not text.strip():
        return ""

    words = text.split()

    if len(words) <= max_len:
        return text

    if len(words) > 2000:
        text = simple_text_truncation(text, 1500)
        words = text.split()

    if len(words) > max_len:
        try:
            if len(words) < 50:
                return text

            input_length = len(words)
            min_length = max(30, min(100, input_length // 5))
            max_length = max(min_length + 50, min(512, input_length // 3))
            if max_length <= min_length:
                max_length = min_length + 50

            summary = summarizer(
                text,
                max_length=max_length,
                min_length=min_length,
                do_sample=False,
                clean_up_tokenization_spaces=True,
                no_repeat_ngram_size=3
            )

            return summary[0]['summary_text']

        except Exception as e:
            print(f"Summarization failed: {e}")
            return simple_text_truncation(text, max_len)

    return text


In [73]:
def compare_claim_with_evidence(claim, evidence):
    """Runs NLI model to compare claim and evidence."""
    if not evidence or not evidence.strip():
        return {
            "claim": claim,
            "label": "Wrong",
            "confidence": 0.0,
            "nli_label": "No Evidence"
        }

    processed_evidence = summarize_text(evidence)

    try:
        inputs = tokenizer.encode_plus(
            processed_evidence,
            claim,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True
        )

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=1)

            nli_labels = ["contradiction", "neutral", "entailment"]
            label_index = torch.argmax(probs).item()
            confidence = probs[0][label_index].item()
            nli_label = nli_labels[label_index]

            if nli_label == "entailment" and confidence > 0.75:
                final_label = "Strong"
            elif nli_label == "entailment" and confidence > 0.6:
                final_label = "Weak"
            elif nli_label == "neutral" and confidence > 0.7:
                final_label = "Weak"
            elif nli_label == "contradiction" and confidence > 0.7:
                final_label = "Wrong"
            else:
                final_label = "Weak"

            return {
                "claim": claim,
                "label": final_label,
                "confidence": round(confidence, 3),
                "nli_label": nli_label
            }

    except Exception as e:
        print(f"NLI comparison failed: {e}")
        return {
            "claim": claim,
            "label": "Wrong",
            "confidence": 0.0,
            "nli_label": "Error"
        }

In [74]:
def aggregate_results(results):
    """
    Aggregates multiple results with improved logic:
    - Considers both label frequency and confidence
    - Uses weighted scoring for final decision
    """
    if not results:
        return "Wrong", 0.0

    label_counts = {"Strong": 0, "Weak": 0, "Wrong": 0}
    confidences_by_label = {"Strong": [], "Weak": [], "Wrong": []}

    for res in results:
        label = res['label']
        confidence = res['confidence']
        label_counts[label] += 1
        confidences_by_label[label].append(confidence)

    weighted_scores = {}
    for label in ["Strong", "Weak", "Wrong"]:
        if confidences_by_label[label]:
            avg_conf = sum(confidences_by_label[label]) / len(confidences_by_label[label])
            weighted_scores[label] = label_counts[label] * avg_conf
        else:
            weighted_scores[label] = 0.0

    total_results = len(results)
    strong_ratio = label_counts["Strong"] / total_results
    wrong_ratio = label_counts["Wrong"] / total_results

    if strong_ratio >= 0.3 and weighted_scores["Strong"] > weighted_scores["Wrong"]:
        final_label = "Strong"
        avg_confidence = sum(confidences_by_label["Strong"]) / len(confidences_by_label["Strong"])
    elif wrong_ratio >= 0.6 and weighted_scores["Wrong"] > weighted_scores["Strong"]:
        final_label = "Wrong"
        avg_confidence = sum(confidences_by_label["Wrong"]) / len(confidences_by_label["Wrong"])
    else:
        final_label = "Weak"
        if confidences_by_label["Weak"]:
            avg_confidence = sum(confidences_by_label["Weak"]) / len(confidences_by_label["Weak"])
        else:
            all_confidences = [res['confidence'] for res in results]
            avg_confidence = sum(all_confidences) / len(all_confidences)

    return final_label, round(avg_confidence, 3)

In [75]:
def fact_check_claim_with_links(claim, urls, max_sources=10):
    """
    Fact-checks a claim against multiple URLs with improved error handling.
    """
    evidence_results = []
    processed_urls = 0

    for url in urls[:max_sources]:
        try:
            print(f"Processing: {url}")
            evidence_text = fetch_text_from_url(url)

            if evidence_text:
                result = compare_claim_with_evidence(claim, evidence_text)
                evidence_results.append(result)
                processed_urls += 1
            else:
                print(f"No text extracted from {url}")

        except Exception as e:
            print(f"Error processing {url}: {e}")

    if not evidence_results:
        return {
            "claim": claim,
            "final_label": "Wrong",
            "average_confidence": 0.0,
            "detailed_results": [],
            "sources_processed": 0,
            "error": "No evidence could be extracted from provided URLs"
        }

    final_label, avg_confidence = aggregate_results(evidence_results)

    return {
        "claim": claim,
        "final_label": final_label,
        "average_confidence": avg_confidence,
        "detailed_results": evidence_results,
        "sources_processed": processed_urls,
        "total_sources": len(urls)
    }

In [76]:
if __name__ == "__main__":
    claim = "Bananas are vegetables."
    urls = [
        "https://www.fao.org/markets-and-trade/commodities-overview/bananas-tropical-fruits/bananas/en",
        "https://www.bananalink.org.uk/all-about-bananas/",
        "https://www.fairtrade.net/en/products/Fairtrade_products/Bananas.html",
        "https://www.britannica.com/plant/banana-plant",
        "https://nhb.gov.in/report_files/banana/BANANA.htm",
        "https://www.healthline.com/nutrition/foods/bananas",
        "https://www.webmd.com/food-recipes/health-benefits-bananas",
        "https://www.medanta.org/patient-education-blog/15-health-benefits-of-raw-bananas-and-why-you-should-eat-them",
        "https://www.livescience.com/45005-banana-nutrition-facts.html",
        "https://mydiagnostics.in/blogs/nutritional/the-best-banana-nutrition-health-benefits-and-facts-you-need-to-know",
        "https://www.growveg.com/plants/us-and-canada/how-to-grow-banana/"
    ]

    result = fact_check_claim_with_links(claim, urls)
    print("\n" + "="*50)
    print("FACT CHECK RESULT")
    print("="*50)
    print(f"Claim: {result['claim']}")
    print(f"Final Label: {result['final_label']}")
    print(f"Average Confidence: {result['average_confidence']}")
    print(f"Sources Processed: {result['sources_processed']}/{result['total_sources']}")
    print("\nDetailed Results:")
    for i, detail in enumerate(result['detailed_results'], 1):
        print(f"  {i}. Label: {detail['label']}, Confidence: {detail['confidence']}, NLI: {detail['nli_label']}")

Processing: https://www.fao.org/markets-and-trade/commodities-overview/bananas-tropical-fruits/bananas/en
Summarization failed: index out of range in self
Processing: https://www.bananalink.org.uk/all-about-bananas/
Summarization failed: index out of range in self
Processing: https://www.fairtrade.net/en/products/Fairtrade_products/Bananas.html
Summarization failed: index out of range in self
Processing: https://www.britannica.com/plant/banana-plant
Summarization failed: index out of range in self
Processing: https://nhb.gov.in/report_files/banana/BANANA.htm
Summarization failed: index out of range in self
Processing: https://www.healthline.com/nutrition/foods/bananas
Summarization failed: index out of range in self
Processing: https://www.webmd.com/food-recipes/health-benefits-bananas
Summarization failed: index out of range in self
Processing: https://www.medanta.org/patient-education-blog/15-health-benefits-of-raw-bananas-and-why-you-should-eat-them
Summarization failed: index out o

In [80]:
pip install -U datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 

In [1]:
from datasets import load_dataset

fever_dataset = load_dataset("fever", "v1.0", split="labelled_dev")

fever_dataset[0]


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'id': 91198,
 'label': 'NOT ENOUGH INFO',
 'claim': 'Colin Kaepernick became a starting quarterback during the 49ers 63rd season in the National Football League.',
 'evidence_annotation_id': 108548,
 'evidence_id': -1,
 'evidence_wiki_url': '',
 'evidence_sentence_id': -1}